# 04 Normalisation, Weighting & DPI Construction


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path

In [2]:
PROCESSED_DIR = Path('../data/processed')
OUTPUT_DIR    = Path('../outputs')

df = pd.read_csv(PROCESSED_DIR / 'driver_season_clustered.csv')

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))

Shape: (105, 19)

Columns: ['season', 'driver_id', 'driver_code', 'driver_name', 'constructor', 'races', 'total_points', 'points_per_race', 'avg_finish_position', 'finish_position_std', 'wins', 'podiums', 'dnfs', 'dnf_rate', 'avg_positions_gained', 'points_finish_rate', 'avg_quali_position', 'cluster', 'cluster_label']


## Normalisation

MinMax normalisation is applied to bring all features to a 0-1 scale 
so they can be combined into a single index. Before scaling, four features 
are inverted because a lower value represents better performance. Without 
this step, a driver with a high DNF rate would score well on that feature 
when in reality it represents poor reliability.

| Feature | Why inverted |
|---|---|
| avg_finish_position | Position 1 is best - lower number is better |
| avg_quali_position | Position 1 is best - lower number is better |
| dnf_rate | Fewer retirements is better |
| finish_position_std | Lower variance means more consistent performance |

In [4]:
index_features = [
    'points_per_race',
    'avg_finish_position',
    'finish_position_std',
    'wins',
    'podiums',
    'points_finish_rate',
    'dnf_rate',
    'avg_positions_gained',
    'avg_quali_position'
]

norm_df = df.copy()

for col in ['avg_finish_position', 'avg_quali_position',
            'dnf_rate', 'finish_position_std']:
    norm_df[col] = -norm_df[col]

scaler = MinMaxScaler()
norm_df[index_features] = scaler.fit_transform(norm_df[index_features])

print("Normalised feature ranges:")
print(norm_df[index_features].describe().loc[['min', 'max']].round(3))

Normalised feature ranges:
     points_per_race  avg_finish_position  finish_position_std  wins  podiums  \
min              0.0                  0.0                  0.0   0.0      0.0   
max              1.0                  1.0                  1.0   1.0      1.0   

     points_finish_rate  dnf_rate  avg_positions_gained  avg_quali_position  
min                 0.0       0.0                   0.0                 0.0  
max                 1.0       1.0                   1.0                 1.0  
